In [1]:
import os
import sys
src_dir = "../../src"
if src_dir not in sys.path:
    sys.path.append(src_dir)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import axes3d
from datetime import datetime
import seaborn as sns
import random
from tqdm import tqdm
from enum import Enum
from sklearn.metrics import mean_squared_error
from filterpy.monte_carlo import (
    multinomial_resample, residual_resample, systematic_resample, stratified_resample
)
from utils.error_report import get_error_from_list, get_error_report
from utils import normalize_angles, lla_to_enu, lla_to_ned

from data_loader import UAV_DataLoader
from configs.configs import MeasurementDataEnum, SetupEnum, SamplingEnum, ErrorEnum
import scipy.stats as stats
from scipy.stats import multivariate_normal
from kalman_filters.extended_kalman_filter import ExtendedKalmanFilter
from kalman_filters.unscented_kalman_filter import UnscentedKalmanFilter
from kalman_filters.ensemble_kalman_filter import EnsembleKalmanFilter
from kalman_filters.particle_filter import ParticleFilter, ResamplingAlgorithms
from kalman_filters.cubature_kalman_filter import CubatureKalmanFilter
from numpy.linalg import norm
from decimal import Decimal, getcontext
from ahrs.filters import Madgwick

%matplotlib inline
np.random.seed(777)

In [2]:
root_path = "../../"

loader = UAV_DataLoader(root_path=root_path, sequence_nr="log0005")

In [3]:
df1 = loader.ref_df.loc[
            (loader.ref_df["device"] == "voxl_imu0") | 
            (loader.ref_df["device"] == "px4_vo")]

In [5]:
loader.px4_vo_df

,timestamp,timestamp_sample,position[0],position[1],position[2],q[0],q[1],q[2],q[3],velocity[0],...,orientation_variance[0],orientation_variance[1],orientation_variance[2],velocity_variance[0],velocity_variance[1],velocity_variance[2],pose_frame,velocity_frame,reset_counter,quality
0,100507946,100487913,0.000288,0.000127,-0.097994,0.999835,-0.012847,0.012813,0.000146,-0.001016,...,2.783582e-08,2.046888e-08,6.440975e-08,0.000007,0.000006,0.000006,2,3,1,100
1,100541037,100517640,0.000287,0.000127,-0.097993,0.999836,-0.012846,0.012797,0.000142,-0.000988,...,2.751919e-08,2.045458e-08,6.438769e-08,0.000007,0.000006,0.000006,2,3,1,100
2,100571952,100557298,0.000288,0.000137,-0.097978,0.999833,-0.013001,0.012805,0.000144,-0.001043,...,2.788102e-08,2.056549e-08,6.505108e-08,0.000007,0.000006,0.000006,2,3,1,100
3,100609834,100588008,0.000290,0.000134,-0.097981,0.999834,-0.012998,0.012797,0.000142,-0.001019,...,2.780535e-08,2.048602e-08,6.566834e-08,0.000007,0.000006,0.000006,2,3,1,100
4,100639798,100617719,0.000295,0.000134,-0.097978,0.999832,-0.013101,0.012823,0.000142,-0.000810,...,2.785921e-08,2.040802e-08,6.585695e-08,0.000007,0.000006,0.000006,2,3,1,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,127343813,127315237,0.212646,-0.246889,-0.970879,0.790699,-0.032414,-0.168347,0.587710,0.052646,...,2.958458e-04,8.123450e-04,1.746018e-04,0.000032,0.000027,0.000036,2,3,1,48
806,127373865,127345048,0.205201,-0.244594,-0.959722,0.806186,-0.028776,-0.164978,0.567466,0.027106,...,2.948090e-04,8.117483e-04,1.749375e-04,0.000028,0.000026,0.000040,2,3,1,40
807,127412707,127384601,0.196228,-0.240682,-0.944662,0.826289,-0.026552,-0.162013,0.538789,0.002152,...,2.927302e-04,8.017901e-04,1.748092e-04,0.000026,0.000025,0.000042,2,3,1,41
808,127439834,127414280,0.190346,-0.240527,-0.932307,0.841537,-0.026280,-0.161480,0.514830,-0.010465,...,2.915222e-04,8.008070e-04,1.751713e-04,0.000025,0.000024,0.000045,2,3,1,33
